In [ ]:
from pathlib import Path
import os
import sys
import json

REPO_ROOT = (
    Path.cwd() if (Path.cwd() / "diffusion_project").is_dir() else Path.cwd().parent
)
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Change model to "unet"; num_subsets can be 1 or 2.
EXPERIMENT = {
    "model": "dit",
    "dataset": "celeba32",
    "data": {"root": "./data", "image_dir": "./data/celeba32", "download": False},
    "output_root": "./results",
    "subset_size": 10000,
    "num_subsets": 2,
    "run_suffixes": ["", "_b", "_c"],
    "device": "auto",
    "resume": True,
    "preview_every": 10,
    "training": {
        "epochs": 400,
        "batch_size": 32,
        "lr": 0.0002,
        "weight_decay": 0.05,
        "diffusion_steps": 1000,
        "sample_steps": 100,
        "seed": 0,
        "amp": True,
    },
    "architecture": {},
}
# Optional batch override: CONFIG_FILE=configs/training.json
if os.environ.get("CONFIG_FILE"):
    EXPERIMENT = json.loads(Path(os.environ["CONFIG_FILE"]).read_text())


# Neural training

Select DiT or UNet and configure one or two disjoint dataset subsets in the first cell.
`architecture={}` uses the selected model's defaults: DiT has 12 blocks and hidden dimension 512;
UNet has scale widths (128,256,256,512). Both use batch size 32.
Each trial has its own initialization seed and resumes independently.
Checkpoints are saved every epoch and previews every ten epochs.

CelebA requires preprocessed images, a normalized tensor, or a custom dataset provider.
See [methods](../docs/methods.md) for preprocessing and training details.


In [ ]:
from diffusion_project.training import train
from diffusion_project.figures import sample_grid

histories = train(EXPERIMENT, preview_callback=sample_grid)


In [ ]:
import matplotlib.pyplot as plt

for directory, history in histories.items():
    plt.plot(
        [r["epoch"] for r in history],
        [r["loss"] for r in history],
        label=Path(directory).name,
    )
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.legend()
plt.show()
